In [1]:
# Trying to optimize Roman Straylight estimator 

In [16]:
import rosalia as rs
import numpy as np
import os  
from tqdm import tqdm 

In [ ]:
def get_transfer_function(level, SCA, X_label, Y_label):
    ndi_name = os.environ['ROSALIACACHE'] + "/CORE/NDI/RST/ndi_lvl" + str(level) + "/" + "lvl" + str(level) + "_SCA_" + str(SCA) + "_SUB_X" + str(X_label) + "_Y" + str(Y_label) + "_HP.fits"
    transfer_function = rs.roman_WFI_NDI_estimator_direct(ndi_name=ndi_name, ra_stars=[0], dec_stars=[0], ra_point=0, dec_point=0, pa_point=0, SCA=SCA, X_label=X_label, Y_label=Y_label, level=level, verbose=False)

In [ ]:
ra_level_1_stars = np.random.uniform(50, 20, 100000)
dec_level_1_stars = np.random.uniform(-20, 20, 100000)

ra_point = 50
dec_point = -20
pa_point = 30

if True:
    SCA = 1
    level = 1 
    subarray_locations_db = rs.roman.get_subarray_locations(SCA=SCA, verbose=False)
    NDI_labels = subarray_locations_db["NDI_labels"]

    X_label = subarray_locations_db["xlabel"][i]
    Y_label = subarray_locations_db["ylabel"][i]

    # filter_transmission_ref , filter_lambda_ref

    if True: 

        ndi_name_1 = os.environ['ROSALIACACHE'] + "/CORE/NDI/RST/ndi_lvl1/" + "lvl1_SCA_" + str(SCA) + "_SUB_X" + str(X_label) + "_Y" + str(Y_label) + "_TAN.fits"
        ndi_name_2 = os.environ['ROSALIACACHE'] + "/CORE/NDI/RST/ndi_lvl2/" + "lvl2_SCA_" + str(SCA) + "_SUB_X" + str(X_label) + "_Y" + str(Y_label) + "_TAN.fits"
        ndi_name_3 = os.environ['ROSALIACACHE'] + "/CORE/NDI/RST/ndi_lvl3/" + "lvl3_SCA_" + str(SCA) + "_SUB_X" + str(X_label) + "_Y" + str(Y_label) + "_HP.fits"

        if True:
            transfer_level_1 = rs.roman.roman_WFI_NDI_estimator_direct(ndi_name=ndi_name_1, 
                                                                    ra_stars=ra_level_1_stars, dec_stars=dec_level_1_stars,
                                                                    ra_point=ra_point, dec_point=dec_point, pa_point=pa_point,
                                                                    SCA=SCA, X_label=X_label, Y_label=Y_label, 
                                                                    level=1, verbose=True)
        if True:
            transfer_level_2 = rs.roman.roman_WFI_NDI_estimator_direct(ndi_name=ndi_name_2, 
                                                                    ra_stars=ra_level_1_stars, dec_stars=dec_level_1_stars,
                                                                    ra_point=ra_point, dec_point=dec_point, pa_point=pa_point,
                                                                    SCA=SCA, X_label=X_label, Y_label=Y_label, 
                                                                    level=2, verbose=True)

        if True:
            transfer_level_3 = rs.roman.roman_WFI_NDI_estimator_direct(ndi_name=ndi_name_3, 
                                                                    ra_stars=ra_level_1_stars, dec_stars=dec_level_1_stars,
                                                                    ra_point=ra_point, dec_point=dec_point, pa_point=pa_point,
                                                                    SCA=SCA, X_label=X_label, Y_label=Y_label, 
                                                                    level=3, verbose=True)

        # Estimate the straylight level from the transfer function and the irradiance of the stars.
        NDI_level_1 = transfer_level_1/superpixel_mm2_area
        NDI_level_2 = transfer_level_2/superpixel_mm2_area
        NDI_level_3 = transfer_level_3/superpixel_mm2_area

        straylight_level_1 = (NDI_level_1*(pixsize**2)*filter_transmission_ref*filter_lambda_ref*irradiance_level_1_stars/const.c/const.h).decompose()
        straylight_level_2 = (NDI_level_2*(pixsize**2)*filter_transmission_ref*filter_lambda_ref*irradiance_level_2_stars/const.c/const.h).decompose()
        straylight_level_3 = (NDI_level_3*(pixsize**2)*filter_transmission_ref*filter_lambda_ref*irradiance_level_3_stars/const.c/const.h).decompose()

        total_straylight_level_1 = bn.nansum(straylight_level_1)
        total_straylight_level_2 = bn.nansum(straylight_level_2)
        total_straylight_level_3 = bn.nansum(straylight_level_3)

        stray_main_offender_level_1 = bn.nanmax(straylight_level_1)
        stray_main_offender_level_2 = bn.nanmax(straylight_level_2)
        stray_main_offender_level_3 = bn.nanmax(straylight_level_3)

        # Once we have the straylight level, we can estimate: 
        # What is the star that produces the maximum straylight level on its own?
        # A.K.A. the main offender.

        where_max_stray_1 = np.where(straylight_level_1.value == stray_main_offender_level_1)[0][0]
        where_max_stray_2 = np.where(straylight_level_2.value == stray_main_offender_level_2)[0][0]
        where_max_stray_3 = np.where(straylight_level_3.value == stray_main_offender_level_3)[0][0]

        id_main_offender_level_1        = id_level_1_stars[where_max_stray_1]
        id_main_offender_level_2        = id_level_2_stars[where_max_stray_2]
        id_main_offender_level_3        = id_level_3_stars[where_max_stray_3]

        source_id_main_offender_level_1 = source_id_level_1[where_max_stray_1]
        source_id_main_offender_level_2 = source_id_level_2[where_max_stray_2]
        source_id_main_offender_level_3 = source_id_level_3[where_max_stray_3]

        max_NDI_level_1                 = NDI_level_1[where_max_stray_1]
        max_NDI_level_2                 = NDI_level_2[where_max_stray_2]
        max_NDI_level_3                 = NDI_level_3[where_max_stray_3]

        # Combine the three levels of straylight and identify the main offender across all levels.
        main_offender_level_123 = np.array([stray_main_offender_level_1,
                                            stray_main_offender_level_2,
                                            stray_main_offender_level_3])

        id_main_offender_level_123 = np.array([id_main_offender_level_1,
                                                id_main_offender_level_2,
                                                id_main_offender_level_3])

        source_name_main_offence_level_123 = np.array([source_id_main_offender_level_1,
                                                        source_id_main_offender_level_2,
                                                        source_id_main_offender_level_3])


        main_offender_SCA[ymin:ymax, xmin:xmax] = id_main_offender_level_123[main_offender_level_123 == bn.nanmax(main_offender_level_123)][0]
        col_stray[i] = straylight_SCA[ymid, xmid]
        col_main_off_id[i] = main_offender_SCA[ymid, xmid]

    if False:
        stray_main_offender_level_1 = 0
        straylight_level_1 = 0/u.s
        id_main_offender_level_1 = 0
        max_NDI_level_1 = 0
        source_id_main_offender_level_1 = None

    # -------------------------------------------------------------------------- #
    # 3 - Combine all the values                                                 #
    # -------------------------------------------------------------------------- #


  0%|          | 0/64 [00:00<?, ?it/s]

100%|██████████| 64/64 [00:19<00:00,  3.36it/s]


In [32]:
from astropy.io import fits
import astropy.wcs as astropy_wcs
from scipy.interpolate import RegularGridInterpolator

ndi_fits = fits.open("/Users/aborlaff/NASA/ROSALIA_DEPOT/ndi_lvl1/lvl1_SCA_1_SUB_X-2.56_Y-2.56_TAN.fits")
w = astropy_wcs.WCS(header=ndi_fits[0].header, fobj=ndi_fits, naxis=2)
x_grid = np.linspace(0, ndi_fits[0].header["NAXIS1"]-1, ndi_fits[0].header["NAXIS1"])
y_grid = np.linspace(0, ndi_fits[0].header["NAXIS2"]-1, ndi_fits[0].header["NAXIS2"])
f = RegularGridInterpolator((x_grid, y_grid), np.flip(ndi_fits[0].data, axis=1).T, bounds_error=False, fill_value=0)

import pickle

data = {"wcs": w, "ndi_interpolator": f}

with open("ndi_test.pkl", "wb") as f:
    pickle.dump(data, f)

In [37]:
with open("/Users/aborlaff/NASA/rosalia_cache/CORE/NDI/RST/ndi_lvl2/lvl2_SCA_1.pkl", "rb") as f:
    data2 = pickle.load(f)

print(data2)

{'NDI_labels': ['X-17.92_Y-17.92', 'X-17.92_Y-12.8', 'X-17.92_Y-7.68', 'X-17.92_Y-2.56', 'X-17.92_Y2.56', 'X-17.92_Y7.68', 'X-17.92_Y12.8', 'X-17.92_Y17.92', 'X-12.8_Y-17.92', 'X-12.8_Y-12.8', 'X-12.8_Y-7.68', 'X-12.8_Y-2.56', 'X-12.8_Y2.56', 'X-12.8_Y7.68', 'X-12.8_Y12.8', 'X-12.8_Y17.92', 'X-7.68_Y-17.92', 'X-7.68_Y-12.8', 'X-7.68_Y-7.68', 'X-7.68_Y-2.56', 'X-7.68_Y2.56', 'X-7.68_Y7.68', 'X-7.68_Y12.8', 'X-7.68_Y17.92', 'X-2.56_Y-17.92', 'X-2.56_Y-12.8', 'X-2.56_Y-7.68', 'X-2.56_Y-2.56', 'X-2.56_Y2.56', 'X-2.56_Y7.68', 'X-2.56_Y12.8', 'X-2.56_Y17.92', 'X2.56_Y-17.92', 'X2.56_Y-12.8', 'X2.56_Y-7.68', 'X2.56_Y-2.56', 'X2.56_Y2.56', 'X2.56_Y7.68', 'X2.56_Y12.8', 'X2.56_Y17.92', 'X7.68_Y-17.92', 'X7.68_Y-12.8', 'X7.68_Y-7.68', 'X7.68_Y-2.56', 'X7.68_Y2.56', 'X7.68_Y7.68', 'X7.68_Y12.8', 'X7.68_Y17.92', 'X12.8_Y-17.92', 'X12.8_Y-12.8', 'X12.8_Y-7.68', 'X12.8_Y-2.56', 'X12.8_Y2.56', 'X12.8_Y7.68', 'X12.8_Y12.8', 'X12.8_Y17.92', 'X17.92_Y-17.92', 'X17.92_Y-12.8', 'X17.92_Y-7.68', 'X17.92_Y-

In [39]:
len(data2["wcs"])

64

In [41]:
data2.keys()

dict_keys(['NDI_labels', 'wcs', 'ndi_interpolator'])

In [10]:
subarray_locations_db

{'subarray_index': array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
        17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
        34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
        51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63]),
 'mask': array([[ 0.,  0.,  0., ...,  7.,  7.,  7.],
        [ 0.,  0.,  0., ...,  7.,  7.,  7.],
        [ 0.,  0.,  0., ...,  7.,  7.,  7.],
        ...,
        [56., 56., 56., ..., 63., 63., 63.],
        [56., 56., 56., ..., 63., 63., 63.],
        [56., 56., 56., ..., 63., 63., 63.]], shape=(4088, 4088)),
 'xlabel': array([-17.92, -17.92, -17.92, -17.92, -17.92, -17.92, -17.92, -17.92,
        -12.8 , -12.8 , -12.8 , -12.8 , -12.8 , -12.8 , -12.8 , -12.8 ,
         -7.68,  -7.68,  -7.68,  -7.68,  -7.68,  -7.68,  -7.68,  -7.68,
         -2.56,  -2.56,  -2.56,  -2.56,  -2.56,  -2.56,  -2.56,  -2.56,
          2.56,   2.56,   2.56,   2.56,   2.56,   2.56,   2.56,   2.56,
         